##Silver Layer Injestion

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

###Importing Libraries

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, min as spark_min, max as spark_max, count, avg, datediff, when,current_timestamp
from pyspark.sql import Window
import pyspark.sql.functions as F

### Orders Table Data Manipulation and Cleaning

In [0]:
df_orders_bronze = spark.table("olist_ecommerce_project.bronze.brz_orders")

# Basic profiling
print("Total rows:", df_orders_bronze.count())
print("Distinct order_id:", df_orders_bronze.select("order_id").distinct().count())

# Null check
df_orders_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_orders_bronze.columns
]).show()

# Check distinct order statuses
df_orders_bronze.select("order_status").distinct().show(truncate=False)

#### Checking & Fixing Null Values

In [0]:
# How many 'delivered' orders have null delivery date? That's a real data issue
print("Delivered orders with null customer delivery date:")
df_orders_bronze.filter(
    (col("order_status") == "delivered") &
    (col("order_delivered_customer_date").isNull())
).count()

# Also check status breakdown of the 2965 null delivery date rows
print("\nStatus breakdown of orders with null delivery date:")
df_orders_bronze.filter(
    col("order_delivered_customer_date").isNull()
).groupBy("order_status").count().orderBy("count", ascending=False).show(truncate=False)

Checking the null values of order_status with delivered value as they shouldn't have null delivery date

In [0]:
df_orders_bronze.filter(
    (col("order_status") == "delivered") &
    (col("order_delivered_customer_date").isNull())
).select(
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
).show(truncate=False)

In [0]:
# Step 1: Flag the 8 problematic 'delivered' orders with missing delivery date
df_orders_silver = df_orders_bronze.withColumn(
    "is_delivery_date_missing",
    when(
        (col("order_status") == "delivered") &
        (col("order_delivered_customer_date").isNull()),
        True
    ).otherwise(False)
)

# Step 2: Calculate delivery delay in days
# Positive = delivered late, Negative = delivered early, Null = not delivered yet
df_orders_silver = df_orders_silver.withColumn(
    "delivery_delay_days",
    when(
        col("order_delivered_customer_date").isNotNull() &
        col("order_estimated_delivery_date").isNotNull(),
        datediff(
            col("order_delivered_customer_date"),
            col("order_estimated_delivery_date")
        )
    ).otherwise(None)
)

# Step 3: Add is_late flag
# True = delivered after estimated date
# False = delivered on time or early
# Null = not yet delivered
df_orders_silver = df_orders_silver.withColumn(
    "is_late",
    when(col("delivery_delay_days").isNull(), None)
    .when(col("delivery_delay_days") > 0, True)
    .otherwise(False)
)

# Step 4: Drop source file audit column
df_orders_silver = df_orders_silver.drop("_source_file")

# Sanity check before writing
print("Total rows:", df_orders_silver.count())
print("\nFlagged rows:", df_orders_silver.filter(col("is_delivery_date_missing") == True).count())
print("\nLate deliveries:", df_orders_silver.filter(col("is_late") == True).count())
print("On time deliveries:", df_orders_silver.filter(col("is_late") == False).count())
print("Not yet delivered:", df_orders_silver.filter(col("is_late").isNull()).count())

# Preview
df_orders_silver.select(
    "order_id",
    "order_status",
    "delivery_delay_days",
    "is_late",
    "is_delivery_date_missing"
).show(10, truncate=False)

##### Creating the Orders Silver Table

In [0]:
(
    df_orders_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.silver.slv_orders")
)

print("slv_orders written successfully")

## Orders — Silver Layer Cleaning Notes

The Orders table is the central table of the Olist dataset, linking customers, 
payments, reviews, and order items together. The following cleaning and enrichment 
steps were applied:

- **Null delivery dates:** Legitimate nulls were left as-is since they reflect 
  real order lifecycle states (e.g., canceled, shipped, processing orders 
  naturally have no delivery date yet).

- **Flagged inconsistencies:** 8 orders marked as `delivered` but missing 
  `order_delivered_customer_date` were flagged with `is_delivery_date_missing = True` 
  rather than dropped, preserving data while marking it as unreliable for 
  delivery KPI calculations.

- **Delivery delay calculation:** A `delivery_delay_days` column was added using 
  the difference between `order_delivered_customer_date` and 
  `order_estimated_delivery_date`. Positive values indicate late delivery, 
  negative values indicate early delivery, and NULL means the order has not 
  been delivered yet.

- **Late delivery flag:** An `is_late` boolean column was added to directly 
  support on-time delivery rate calculations in the Gold layer.

### New columns added and their Gold layer use cases

| Column | Gold Use Case |
|---|---|
| `is_delivery_date_missing` | Filter unreliable rows out of delivery KPIs |
| `delivery_delay_days` | Average delay analysis per seller, state, and category |
| `is_late` | On-time delivery rate — a key logistics KPI |